# Job Scraper ETL — Jupyter Notebook

**Goal:** Perform ETL (Extract, Transform, Load) on job listings data scraped into SQLite, clean the data, and export to CSV.
**Tech stack:** Python, pandas, sqlite3
**Pipeline Steps:**
1. Extract data from SQLite database.
2. Transform (clean, standardize text, remove duplicates, handle missing values).
3. Load cleaned data into CSV for further analysis or dashboarding.

In [7]:
import sqlite3
import pandas as pd
import re
import logging


# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


# Database and file paths
DB_PATH = 'jobs.db'
OUTPUT_CSV = 'cleaned_jobs.csv'

## 1) Extract — Read Data from SQLite

In [8]:
def extract_data(db_path: str = DB_PATH) -> pd.DataFrame:
    logging.info(f"Extracting all available data from {db_path} ...")
    conn = sqlite3.connect(db_path)
    # Select all columns to capture new or future fields
    df = pd.read_sql_query('SELECT * FROM jobs', conn)
    conn.close()
    logging.info(f"Loaded {len(df)} records with {len(df.columns)} columns from database.")
    return df

## 2) Transform — Clean & Standardize Data

In [9]:
def clean_text(text: str) -> str:
	if pd.isna(text):
		return text
	text = re.sub(r'\s+', ' ', text) # remove multiple spaces
	text = text.strip()
	return text

In [10]:
def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    logging.info("Starting data cleaning and transformation ...")

    # Drop duplicates
    df = df.drop_duplicates(subset=['title', 'company', 'location'], keep='first')

    # Clean text
    text_cols = ['title', 'company', 'location', 'summary', 'post_date']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).apply(clean_text)

    # Handle missing
    df = df.fillna({'company': 'Unknown', 'location': 'Unspecified'})

    # Extract job domain keywords
    df['domain'] = df['title'].apply(lambda x: 'Data' if 'data' in x.lower() else
                                                'Engineering' if 'engineer' in x.lower() else
                                                'Business' if 'business' in x.lower() else
                                                'Other')

    # Extract salary info if available in summary
    df['salary_mentioned'] = df['summary'].str.contains('EGP|USD|salary|جنيه', case=False, na=False)

    # Sort by most recent
    if 'post_date' in df.columns:
        df['post_date'] = df['post_date'].replace('Just posted', '0 days ago')
        df['days_ago'] = df['post_date'].str.extract(r'(\\d+)').astype(float)
        df = df.sort_values('days_ago', ascending=True)

    logging.info(f"Cleaned dataset shape: {df.shape}")
    return df.reset_index(drop=True)


## 3) Load — Export Cleaned Data to CSV

In [11]:
def load_to_csv(df: pd.DataFrame, path: str = OUTPUT_CSV):
	df.to_csv(path, index=False, encoding='utf-8-sig')
	logging.info(f"Cleaned data exported successfully to {path}")

## 4) ETL Pipeline Runner

In [12]:
def run_etl(db_path: str = DB_PATH, output_csv: str = OUTPUT_CSV):
	df_raw = extract_data(db_path)
	df_clean = transform_data(df_raw)
	load_to_csv(df_clean, output_csv)
	return df_clean

## 5) Example Run
Uncomment the next cell to execute the full ETL pipeline.

In [15]:
# Ensure the 'jobs' table exists with sample data
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Create table if it doesn't exist
cursor.execute('''
CREATE TABLE IF NOT EXISTS jobs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    company TEXT,
    location TEXT,
    summary TEXT,
    post_date TEXT
)
''')

# Insert sample data if table is empty
pd.read_sql_query('SELECT * FROM jobs', conn)

# Now run the ETL pipeline
df_cleaned = run_etl(DB_PATH, OUTPUT_CSV)
df_cleaned.head()

2025-10-14 12:43:49,128 - INFO - Extracting all available data from jobs.db ...
2025-10-14 12:43:49,131 - INFO - Loaded 3 records with 6 columns from database.
2025-10-14 12:43:49,132 - INFO - Starting data cleaning and transformation ...
2025-10-14 12:43:49,139 - INFO - Cleaned dataset shape: (3, 9)
2025-10-14 12:43:49,141 - INFO - Cleaned data exported successfully to cleaned_jobs.csv


,id,title,company,location,summary,post_date,domain,salary_mentioned,days_ago
0,1,Data Scientist,TechCorp,New York,Analyze data,2 days ago,Data,False,NaN
1,2,Software Engineer,InnovateX,San Francisco,Develop software,0 days ago,Engineering,False,NaN
2,3,Analyst,BizGroup,Chicago,Business analysis,5 days ago,Other,False,NaN


## 6) Next Steps & Notes
- Add normalization for job titles and locations using lookup tables.
- Integrate with Power BI or Tableau for visualization.
- Automate ETL scheduling via Airflow, cron, or a cloud function.
- Add validation rules (e.g., exclude expired jobs).